In [47]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary
from torchmetrics import Accuracy
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.optim as optim

import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import optuna
from io import BytesIO
import csv

Dataset preparation

In [48]:

# Import the dataset

df = pd.read_csv('Processed_Randomized_data_with_single_coil_measurements_Nov_22.csv', parse_dates = True, index_col=0)  #  Coil data

df = df.dropna()

# Sample data
X = df[['cmdCoilCurrent_1(A)','cmdCoilCurrent_2(A)','cmdCoilCurrent_3(A)','cmdCoilCurrent_4(A)',
        'cmdCoilCurrent_5(A)','cmdCoilCurrent_6(A)','cmdCoilCurrent_7(A)','cmdCoilCurrent_8(A)',
        'x', 'y', 'z']].values  # input current and position
y = df[['U','V','W']].values # label


# split the data into train and test

X_train_and_val, X_test, y_train_and_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42) # 15% for test

# Split data (90% train, 10% validation)
X_train, X_val, y_train, y_val = train_test_split(X_train_and_val, y_train_and_val, test_size=0.15, random_state=42) # 15% for validation

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)



# Create DataLoader for training and validation
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
# test_loader = DataLoader(test_dataset,batch_size=64, shuffle=False)

C:\Users\Micro Lab ML\AppData\Local\Temp\ipykernel_21484\1000395435.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv('Processed_Randomized_data_with_single_coil_measurements_Nov_22.csv', parse_dates = True, index_col=0)  #  Coil data


In [49]:
class WaveAct(nn.Module):
    def __init__(self):
        super(WaveAct, self).__init__() 
        self.w1 = nn.Parameter(torch.ones(1), requires_grad=True)
        self.w2 = nn.Parameter(torch.ones(1), requires_grad=True)

    def forward(self, x):
        return self.w1 * torch.sin(x) + self.w2 * torch.cos(x)

In [50]:

def build_model(n_layers, hidden_units, activation):
    """
    Dynamically builds a feedforward neural network based on given parameters.

    Args:
        n_layers (int): Number of hidden layers.
        hidden_units (int): Number of units in each hidden layer.
        activation (str): Activation function to use ("relu" or "tanh").

    Returns:
        torch.nn.Module: The dynamically constructed neural network model.
    """
    # Choose the activation function
    if activation == "leaky_ReLU":
        activation_fn = nn.LeakyReLU()
    elif activation == "ReLU":
        activation_fn = nn.ReLU()
    else:
        raise ValueError(f"Unsupported activation function: {activation}")

    # Define the network layers
    layers = []
    input_dim = 11  # input current and position (x,y,z)
    output_dim = 3  # Bx, By,Bz - magnetic field prediction

    # Input layer
    layers.append(nn.Linear(input_dim, hidden_units))
    layers.append(activation_fn)

    # Hidden layers
    for _ in range(n_layers - 1):
        layers.append(nn.Linear(hidden_units, hidden_units))
        layers.append(activation_fn)

    # Output layer
    layers.append(nn.Linear(hidden_units, output_dim))

    # Combine all layers into a sequential model
    model = nn.Sequential(*layers)

    return model

Implement the training loop

In [51]:
# Get cpu or gpu for training.
device = "cuda" if torch.cuda.is_available() else "cpu"

In [52]:
def metrics_fn(outputs, labels):

    MAE = torch.mean(torch.abs(labels-outputs),dim= 0, keepdim=True)
    labels = torch.abs(torch.where(labels == 0, torch.tensor(1e-6), labels)) # to avoid division by zero
    labels = torch.mean(torch.abs(labels), dim = 0, keepdim = True)
        
    MAEP = MAE/labels
    MAE_norm= torch.norm(MAE) # mean absolute error 
    MAEP_norm = MAE_norm/torch.norm(labels) # mean absolute error
    
    return MAE, MAE_norm, MAEP, MAEP_norm

def test_model(test_loader, model, metrics_fn):
  criterion = nn.MSELoss()
  running_test_loss = 0.0
  running_MAE = 0.0
  running_MAE_norm = 0.0
  running_MAEP = 0.0
  running_MAPE_norm = 0.0
  
  with torch.no_grad():
      for inputs, labels in test_loader:
          inputs, labels = inputs.to(device),labels.to(device)
          outputs = model(inputs)
          
          loss = criterion(outputs, labels)
          MAE, MAE_norm, MAEP, MAEP_norm = metrics_fn(outputs, labels)
          
          running_test_loss += loss.item()
          running_MAE += MAE
          running_MAE_norm += MAE_norm.item()
          running_MAEP += MAEP
          running_MAPE_norm += MAEP_norm.item()
          
      running_test_loss = np.sqrt(running_test_loss / len(test_loader)) # RMSE in mT over each Bx, By and Bz direction of all measurement
      running_MAE = running_MAE/ len(test_loader)
      running_MAE_norm = running_MAE_norm/ len(test_loader)
      running_MAEP = running_MAEP/ len(test_loader)
      running_MAPE_norm = running_MAPE_norm/ len(test_loader)
          
  # print(f'Test Error: {running_test_loss:.4f}')
  return running_test_loss, running_MAE, running_MAE_norm, running_MAEP,running_MAPE_norm 
    

setup mlflow

In [53]:
def get_or_create_experiment(experiment_name):
    """
    Retrieve the ID of an existing MLflow experiment or create a new one if it doesn't exist.

    This function checks if an experiment with the given name exists within MLflow.
    If it does, the function returns its ID. If not, it creates a new experiment
    with the provided name and returns its ID.

    Parameters:
    - experiment_name (str): Name of the MLflow experiment.

    Returns:
    - str: ID of the existing or newly created MLflow experiment.
    """

    if experiment := mlflow.get_experiment_by_name(experiment_name):
        return experiment.experiment_id
    else:
        return mlflow.create_experiment(experiment_name)

In [54]:
# mlflow.login() # to connect to databricks servers
mlflow.set_tracking_uri("http://localhost:5000")  # connecting to local host
experiment_id = get_or_create_experiment("FCN_B_Model")

# Set the current active MLflow experiment
mlflow.set_experiment(experiment_id=experiment_id)
run_name = "model_tunning_trial_2"

In [ ]:
# Optuna objective function
def objective(trial):
    n_layers = 6  # Number of hidden layers
    hidden_units = trial.suggest_int("hidden_units", 250, 500)  # Units per layer
    activation = trial.suggest_categorical("activation", ["leaky_ReLU", "ReLU"])  # Activation
    batch_size = trial.suggest_int("batch_size",128,254,step = 64)
    learning_rate = trial.suggest_float("learning_rate", 0.0004, 0.001, log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
    test_loader = DataLoader(test_dataset,batch_size=batch_size, shuffle=False, drop_last=True)

    model = build_model(n_layers, hidden_units, activation).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    epoches = 150
    lowest_val_loss = float('inf') 
    criterion = nn.MSELoss()

    # Start MLflow run for this trial
    with mlflow.start_run(nested=True) as run:

        for epoch in range(epoches):  # Loop over the dataset multiple times
            train_loss = 0.0
            val_loss = 0.0
            model.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                optimizer.zero_grad()  # Zero the parameter gradients
                
                # Forward + backward + optimize
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()

            train_loss = np.sqrt(train_loss / len(train_loader)) # RMSE in mT over each Bx, By and Bz direction of all measurement
            model.eval()
            
            ## model validaiton
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device),labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()        

            val_loss = np.sqrt(val_loss / len(val_loader)) # RMSE in mT over each Bx, By and Bz direction of all measurement
            

            # Log the training and validation losses for this epoch
            mlflow.log_metric("train_loss_RMSE", train_loss, step=epoch)
            mlflow.log_metric("val_loss_RMSE", val_loss, step=epoch)
            
            # Track and log the lowest validation loss
            if val_loss < lowest_val_loss:
                lowest_val_loss = val_loss
                # Log the lowest validation loss at the current epoch
                mlflow.log_metric("lowest_val_loss", lowest_val_loss, step=epoch)
                mlflow.pytorch.log_model(model,"lowest_val_model")
                test_RMSE, test_MAE, test_MAE_norm, test_MAEP, test_MAPE_norm  = test_model(test_loader, model, metrics_fn)

                mlflow.log_metric("test_RMSE",test_RMSE, step = epoch)

                mlflow.log_metric("test_Bx_MAE", test_MAE[0][0].item(), step = epoch)
                mlflow.log_metric("test_By_MAE", test_MAE[0][1].item(), step = epoch)
                mlflow.log_metric("test_Bz_MAE", test_MAE[0][2].item(), step = epoch)

                mlflow.log_metric("test_MAE_norm",test_MAE_norm, step = epoch)

                mlflow.log_metric("test_Bx_MAEP", test_MAEP[0][0].item(), step = epoch)
                mlflow.log_metric("test_By_MAEP", test_MAEP[0][1].item(), step = epoch)
                mlflow.log_metric("test_Bz_MAEP", test_MAEP[0][2].item(), step = epoch)

                mlflow.log_metric("test_MAPE_norm",test_MAPE_norm, step = epoch)



        # Log hyperparameters
        # mlflow.log_param("n_layers", n_layers)
        mlflow.log_param("hidden_units", hidden_units)
        mlflow.log_param("activation", activation)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("learning rate", learning_rate)

    
    return lowest_val_loss.item()

# Optuna study
# Initiate the parent run and call the hyperparameter tuning child run logic
with mlflow.start_run(experiment_id=experiment_id, run_name=run_name, nested=True):
    # Initialize the Optuna study
    study = optuna.create_study(direction="minimize")

    # Execute the hyperparameter optimization trials.
    # Note the addition of the `champion_callback` inclusion to control our logging
    study.optimize(objective, n_trials=80)
    
    # Log tags
    mlflow.set_tags(
        tags={
            "project": "MEng_Project",
            "optimizer_engine": "optuna",
            "optimizer": "Adam",
            "Batch Normalized": "No",
            "model_acchitecture": "FCN",
            "optimization": "model_depth, width and activation_function, bath_size, learning_rate",

        }
    )
    
    # Retrieve the best trial
    
    best_trial = study.best_trial

    # Extract the best hyperparameters
    best_n_layers = best_trial.params["n_layers"]
    best_hidden_units = best_trial.params["hidden_units"]
    best_activation = best_trial.params["activation"]

    # Rebuild the optimized model
    # optimized_model = build_model(best_n_layers, best_hidden_units, best_activation)
    
    # Best trial
    print("tunned n_layer:", best_n_layers)
    print("tunned model width", best_hidden_units)
    print("best activation function", best_activation)
    
    

[I 2024-11-25 06:40:49,210] A new study created in memory with name: no-name-92d44b6d-0713-4ca2-a91c-130b6ff399d8
c:\Users\Micro Lab ML\Desktop\Gediyon\.venv\lib\site-packages\optuna\distributions.py:708: UserWarning: The distribution is specified by [128, 254] and step=64, but the range is not divisible by `step`. It will be replaced by [128, 192].
  warnings.warn(
2024/11/25 06:40:52 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2024/11/25 06:40:55 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installab

🏃 View run resilient-mouse-281 at: http://localhost:5000/#/experiments/128438271774454410/runs/86731d592fe946d1b1fa129b3db34577
🧪 View experiment at: http://localhost:5000/#/experiments/128438271774454410


2024/11/25 06:49:44 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2024/11/25 06:49:47 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2024/11/25 06:49:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/11/25 06:49:50 WARNING mlflow.utils.requirement

🏃 View run defiant-owl-764 at: http://localhost:5000/#/experiments/128438271774454410/runs/4f75acfe3ba54c74a29d7ea1f977057a
🧪 View experiment at: http://localhost:5000/#/experiments/128438271774454410


2024/11/25 06:59:00 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2024/11/25 06:59:03 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2024/11/25 06:59:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/11/25 06:59:06 WARNING mlflow.utils.requirement

...

2024/11/25 16:51:11 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2024/11/25 16:51:14 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2024/11/25 16:51:14 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/11/25 16:51:16 WARNING mlflow.utils.requirement

🏃 View run dashing-calf-841 at: http://localhost:5000/#/experiments/128438271774454410/runs/441a71061d2e43cea0f3ebc95fe42af4
🧪 View experiment at: http://localhost:5000/#/experiments/128438271774454410


2024/11/25 16:59:21 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2024/11/25 16:59:25 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu118) contains a local version label (+cu118). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2024/11/25 16:59:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/11/25 16:59:27 WARNING mlflow.utils.requirement

🏃 View run merciful-roo-756 at: http://localhost:5000/#/experiments/128438271774454410/runs/2a31158cb0e54c02aeab5273d5605ba0
🧪 View experiment at: http://localhost:5000/#/experiments/128438271774454410
🏃 View run model_tunning_trial_2 at: http://localhost:5000/#/experiments/128438271774454410/runs/26324785ef4e49239b93a80126582705
🧪 View experiment at: http://localhost:5000/#/experiments/128438271774454410


KeyError: 'n_layers'